In [ ]:
# ============================================================
# STEP 2: DESCRIPTIVE STATISTICS AND DIAGNOSTIC TESTS
# ============================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.stats import jarque_bera
from statsmodels.tsa.stattools import adfuller
from statsmodels.stats.diagnostic import acorr_ljungbox, het_arch
from statsmodels.graphics.tsaplots import plot_acf
from statsmodels.graphics.gofplots import qqplot

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. Paths
# ------------------------------------------------------------

PROCESSED_DIR = Path("data/processed")
TABLE_DIR = Path("outputs/tables")
FIGURE_DIR = Path("outputs/figures")

TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

RETURNS_PATH = PROCESSED_DIR / "vietnam_size_indices_log_returns_common_20141121_20251231.csv"

# Fallback if running directly in ChatGPT sandbox
if not RETURNS_PATH.exists():
    RETURNS_PATH = Path("/mnt/data/vietnam_size_indices_log_returns_common_20141121_20251231.csv")

# ------------------------------------------------------------
# 2. Load returns
# ------------------------------------------------------------

returns = pd.read_csv(RETURNS_PATH, parse_dates=["date"])
returns = returns.set_index("date").sort_index()

expected_columns = ["VN30", "VNAllshare", "VNMidcap", "VNSmallcap"]

missing_cols = [c for c in expected_columns if c not in returns.columns]
if missing_cols:
    raise ValueError(f"Missing expected return columns: {missing_cols}")

returns = returns[expected_columns]

# Basic validation
assert returns.index.is_monotonic_increasing, "Date index is not sorted."
assert returns.index.duplicated().sum() == 0, "Duplicate dates exist."
assert returns.isna().sum().sum() == 0, "Missing values exist in returns."
assert np.isfinite(returns.to_numpy()).all(), "Infinite values exist in returns."

print("Return dataset loaded successfully.")
print("Sample:", returns.index.min().date(), "to", returns.index.max().date())
print("Shape:", returns.shape)
display(returns.head())
display(returns.tail())

In [ ]:
# ============================================================
# 2.1 DESCRIPTIVE STATISTICS
# ============================================================

def descriptive_stats(y):
    y = y.dropna()

    return pd.Series({
        "n_obs": len(y),
        "mean": y.mean(),
        "median": y.median(),
        "std": y.std(ddof=1),
        "variance": y.var(ddof=1),
        "min": y.min(),
        "q01": y.quantile(0.01),
        "q05": y.quantile(0.05),
        "q25": y.quantile(0.25),
        "q75": y.quantile(0.75),
        "q95": y.quantile(0.95),
        "q99": y.quantile(0.99),
        "max": y.max(),
        "skewness": y.skew(),
        "kurtosis_excess": y.kurt(),
        "kurtosis_pearson": y.kurt() + 3,
        "annualized_mean_pct": y.mean() * 252,
        "annualized_volatility_pct": y.std(ddof=1) * np.sqrt(252),
    })

desc_stats = returns.apply(descriptive_stats).T

desc_stats.to_csv(TABLE_DIR / "step2_descriptive_statistics_returns.csv")

display(desc_stats.round(6))

In [ ]:
# ============================================================
# 2.2 JARQUE-BERA NORMALITY TEST
# H0: returns are normally distributed
# Reject H0 if p-value < 0.05
# ============================================================

jb_rows = []

for col in returns.columns:
    y = returns[col].dropna()
    jb = jarque_bera(y)

    jb_rows.append({
        "index": col,
        "jb_stat": jb.statistic,
        "jb_pvalue": jb.pvalue,
        "reject_normality_5pct": jb.pvalue < 0.05
    })

jb_results = pd.DataFrame(jb_rows)
jb_results.to_csv(TABLE_DIR / "step2_jarque_bera_test.csv", index=False)

display(jb_results.round(6))

In [ ]:
# ============================================================
# 2.3 AUGMENTED DICKEY-FULLER TEST
# H0: unit root / non-stationary
# Reject H0 if p-value < 0.05
# For return series, expected result: stationary
# ============================================================

adf_rows = []

for col in returns.columns:
    y = returns[col].dropna()

    adf_result = adfuller(
        y,
        regression="c",
        autolag="AIC"
    )

    adf_rows.append({
        "index": col,
        "adf_stat": adf_result[0],
        "adf_pvalue": adf_result[1],
        "used_lag": adf_result[2],
        "n_obs": adf_result[3],
        "critical_1pct": adf_result[4]["1%"],
        "critical_5pct": adf_result[4]["5%"],
        "critical_10pct": adf_result[4]["10%"],
        "icbest": adf_result[5],
        "stationary_5pct": adf_result[1] < 0.05
    })

adf_results = pd.DataFrame(adf_rows)
adf_results.to_csv(TABLE_DIR / "step2_adf_stationarity_test.csv", index=False)

display(adf_results.round(6))

In [ ]:
# ============================================================
# 2.4 LJUNG-BOX TEST ON RETURNS
# H0: no autocorrelation up to selected lag
# If p-value < 0.05: serial correlation exists in returns
# ============================================================

LB_LAGS = [10, 20]

lb_return_rows = []

for col in returns.columns:
    y = returns[col].dropna()

    lb = acorr_ljungbox(
        y,
        lags=LB_LAGS,
        return_df=True
    )

    for lag in LB_LAGS:
        lb_return_rows.append({
            "index": col,
            "lag": lag,
            "lb_stat_returns": lb.loc[lag, "lb_stat"],
            "lb_pvalue_returns": lb.loc[lag, "lb_pvalue"],
            "autocorrelation_returns_5pct": lb.loc[lag, "lb_pvalue"] < 0.05
        })

lb_returns_results = pd.DataFrame(lb_return_rows)
lb_returns_results.to_csv(TABLE_DIR / "step2_ljungbox_returns.csv", index=False)

display(lb_returns_results.round(6))

In [ ]:
# ============================================================
# 2.5 LJUNG-BOX TEST ON SQUARED RETURNS
# H0: no autocorrelation in squared returns
# If p-value < 0.05: volatility clustering / second-moment dependence
# ============================================================

lb_sq_rows = []

for col in returns.columns:
    y2 = returns[col].dropna() ** 2

    lb = acorr_ljungbox(
        y2,
        lags=LB_LAGS,
        return_df=True
    )

    for lag in LB_LAGS:
        lb_sq_rows.append({
            "index": col,
            "lag": lag,
            "lb_stat_squared_returns": lb.loc[lag, "lb_stat"],
            "lb_pvalue_squared_returns": lb.loc[lag, "lb_pvalue"],
            "volatility_clustering_5pct": lb.loc[lag, "lb_pvalue"] < 0.05
        })

lb_squared_results = pd.DataFrame(lb_sq_rows)
lb_squared_results.to_csv(TABLE_DIR / "step2_ljungbox_squared_returns.csv", index=False)

display(lb_squared_results.round(6))

In [ ]:
# ============================================================
# 2.6 ARCH-LM TEST
# H0: no ARCH effects
# If p-value < 0.05: ARCH effects exist
# This is direct evidence supporting ARCH/GARCH-type models
# ============================================================

ARCH_LAGS = 10

arch_rows = []

for col in returns.columns:
    y = returns[col].dropna()

    lm_stat, lm_pvalue, f_stat, f_pvalue = het_arch(
        y,
        nlags=ARCH_LAGS
    )

    arch_rows.append({
        "index": col,
        "arch_lags": ARCH_LAGS,
        "arch_lm_stat": lm_stat,
        "arch_lm_pvalue": lm_pvalue,
        "arch_f_stat": f_stat,
        "arch_f_pvalue": f_pvalue,
        "arch_effects_5pct": lm_pvalue < 0.05
    })

arch_results = pd.DataFrame(arch_rows)
arch_results.to_csv(TABLE_DIR / "step2_arch_lm_test.csv", index=False)

display(arch_results.round(6))

In [ ]:
# ============================================================
# 2.7 FINAL SUMMARY: IS GARCH JUSTIFIED?
# ============================================================

# Use lag 10 as the main Ljung-Box diagnostic
lb_ret_10 = lb_returns_results[lb_returns_results["lag"] == 10].copy()
lb_sq_10 = lb_squared_results[lb_squared_results["lag"] == 10].copy()

summary = (
    desc_stats[[
        "n_obs",
        "mean",
        "std",
        "skewness",
        "kurtosis_excess",
        "kurtosis_pearson"
    ]]
    .reset_index()
    .rename(columns={"index": "index"})
)

summary = summary.merge(
    jb_results[["index", "jb_pvalue", "reject_normality_5pct"]],
    on="index",
    how="left"
)

summary = summary.merge(
    adf_results[["index", "adf_pvalue", "stationary_5pct"]],
    on="index",
    how="left"
)

summary = summary.merge(
    lb_ret_10[["index", "lb_pvalue_returns", "autocorrelation_returns_5pct"]],
    on="index",
    how="left"
)

summary = summary.merge(
    lb_sq_10[["index", "lb_pvalue_squared_returns", "volatility_clustering_5pct"]],
    on="index",
    how="left"
)

summary = summary.merge(
    arch_results[["index", "arch_lm_pvalue", "arch_effects_5pct"]],
    on="index",
    how="left"
)

summary["garch_supported"] = (
    summary["stationary_5pct"]
    & summary["reject_normality_5pct"]
    & (
        summary["volatility_clustering_5pct"]
        | summary["arch_effects_5pct"]
    )
)

summary.to_csv(TABLE_DIR / "step2_garch_readiness_summary.csv", index=False)

display(summary.round(6))

In [ ]:
# ============================================================
# 2.8 CORRELATION MATRIX
# Useful for showing co-movement across size segments
# ============================================================

corr_matrix = returns.corr()
corr_matrix.to_csv(TABLE_DIR / "step2_return_correlation_matrix.csv")

display(corr_matrix.round(4))

In [ ]:
# ============================================================
# 2.9 EXPORT ALL STEP 2 TABLES TO ONE EXCEL FILE
# ============================================================

excel_path = TABLE_DIR / "step2_descriptive_statistics_and_diagnostics.xlsx"

with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    desc_stats.to_excel(writer, sheet_name="Descriptive_Stats")
    jb_results.to_excel(writer, sheet_name="Jarque_Bera", index=False)
    adf_results.to_excel(writer, sheet_name="ADF_Test", index=False)
    lb_returns_results.to_excel(writer, sheet_name="LB_Returns", index=False)
    lb_squared_results.to_excel(writer, sheet_name="LB_Squared_Returns", index=False)
    arch_results.to_excel(writer, sheet_name="ARCH_LM", index=False)
    summary.to_excel(writer, sheet_name="GARCH_Readiness", index=False)
    corr_matrix.to_excel(writer, sheet_name="Correlation")

print(f"Saved Excel report to: {excel_path}")

In [ ]:
# ============================================================
# 2.10 PLOTS FOR DESCRIPTIVE ANALYSIS
# ============================================================

# 1. Return time series
for col in returns.columns:
    plt.figure(figsize=(12, 4))
    plt.plot(returns.index, returns[col])
    plt.title(f"{col} daily log returns")
    plt.xlabel("Date")
    plt.ylabel("Log return (%)")
    plt.tight_layout()
    # plt.savefig(FIGURE_DIR / f"step2_{col}_daily_log_returns.png", dpi=300)
    plt.show()

# 2. Histogram
for col in returns.columns:
    plt.figure(figsize=(8, 4))
    plt.hist(returns[col].dropna(), bins=80)
    plt.title(f"{col} return distribution")
    plt.xlabel("Log return (%)")
    plt.ylabel("Frequency")
    plt.tight_layout()
    # plt.savefig(FIGURE_DIR / f"step2_{col}_return_histogram.png", dpi=300)
    plt.show()

# 3. QQ plot
for col in returns.columns:
    plt.figure(figsize=(6, 6))
    qqplot(returns[col].dropna(), line="s", fit=True)
    plt.title(f"{col} Q-Q plot")
    plt.tight_layout()
    # plt.savefig(FIGURE_DIR / f"step2_{col}_qq_plot.png", dpi=300)
    plt.show()

# 4. ACF of returns
for col in returns.columns:
    fig = plt.figure(figsize=(10, 4))
    plot_acf(returns[col].dropna(), lags=40)
    plt.title(f"{col} ACF of returns")
    plt.tight_layout()
    # plt.savefig(FIGURE_DIR / f"step2_{col}_acf_returns.png", dpi=300)
    plt.show()

# 5. ACF of squared returns
for col in returns.columns:
    fig = plt.figure(figsize=(10, 4))
    plot_acf((returns[col].dropna() ** 2), lags=40)
    plt.title(f"{col} ACF of squared returns")
    plt.tight_layout()
    # plt.savefig(FIGURE_DIR / f"step2_{col}_acf_squared_returns.png", dpi=300)
    plt.show()

In [ ]:
# ============================================================
# 2.11 AUTOMATIC INTERPRETATION FOR REPORT WRITING
# ============================================================

interpretation_rows = []

for _, row in summary.iterrows():
    index_name = row["index"]

    normality_text = (
        "rejects normality"
        if row["reject_normality_5pct"]
        else "does not reject normality"
    )

    stationarity_text = (
        "stationary"
        if row["stationary_5pct"]
        else "non-stationary"
    )

    volatility_text = (
        "shows volatility clustering"
        if row["volatility_clustering_5pct"]
        else "does not show volatility clustering"
    )

    arch_text = (
        "has significant ARCH effects"
        if row["arch_effects_5pct"]
        else "does not have significant ARCH effects"
    )

    garch_text = (
        "GARCH-type modeling is supported"
        if row["garch_supported"]
        else "GARCH-type modeling is not strongly supported"
    )

    interpretation_rows.append({
        "index": index_name,
        "interpretation": (
            f"{index_name}: The return series is {stationarity_text}. "
            f"The Jarque-Bera test {normality_text}, indicating non-normality if rejected. "
            f"The Ljung-Box test on squared returns {volatility_text}. "
            f"The ARCH-LM test indicates that the series {arch_text}. "
            f"Overall, {garch_text}."
        )
    })

interpretation_table = pd.DataFrame(interpretation_rows)
interpretation_table.to_csv(TABLE_DIR / "step2_interpretation_text.csv", index=False)

for text in interpretation_table["interpretation"]:
    print(text)
    print()

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

FIG_DIR = Path("figures")
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Optional: make plots publication-friendly
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "axes.grid": True,
    "grid.alpha": 0.3,
})


INDEX_ORDER = ["VN30", "VNAllshare", "VNMidcap", "VNSmallcap"]


def find_date_column(df):
    candidates = ["Date", "date", "trading_date", "TradingDate", "time", "Time"]
    for c in candidates:
        if c in df.columns:
            return c
    raise ValueError(f"No date column found. Available columns: {df.columns.tolist()}")


def standardize_date_index(df):
    df = df.copy()
    date_col = find_date_column(df)
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.sort_values(date_col).set_index(date_col)
    return df


def pick_index_columns(df):
    cols = []
    for idx in INDEX_ORDER:
        exact = [c for c in df.columns if c == idx]
        contains = [c for c in df.columns if idx.lower() in c.lower()]
        if exact:
            cols.append(exact[0])
        elif contains:
            cols.append(contains[0])
        else:
            raise ValueError(f"Cannot find column for {idx}. Available columns: {df.columns.tolist()}")
    return cols


def savefig(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, bbox_inches="tight")
    plt.close()
    print(f"Saved: {path}")

In [ ]:
# ============================================================
# Figure 1: Normalized closing prices
# Input: vietnam_size_indices_close_common_20141120_20251231.csv
# Output: figures/fig_01_normalized_prices.pdf
# ============================================================

close_path = Path("data/processed/vietnam_size_indices_close_common_20141120_20251231.csv")
close_df = pd.read_csv(close_path)
close_df = standardize_date_index(close_df)

close_cols = pick_index_columns(close_df)
close_plot = close_df[close_cols].copy()
close_plot.columns = INDEX_ORDER

# Rebase to 100 at first available observation
norm_prices = close_plot.div(close_plot.iloc[0]).mul(100)

fig, ax = plt.subplots(figsize=(10, 4.8))

# Mark crisis periods first so they stay behind the index lines
ax.axvspan(
    pd.Timestamp("2020-02-01"),
    pd.Timestamp("2020-04-30"),
    color="#D55E00",
    alpha=0.16,
    linewidth=0,
    label="COVID-19 shock",
    zorder=0
)

ax.axvspan(
    pd.Timestamp("2022-01-01"),
    pd.Timestamp("2022-12-31"),
    color="#0072B2",
    alpha=0.14,
    linewidth=0,
    label="2022 sell-off",
    zorder=0
)

for col in INDEX_ORDER:
    ax.plot(
        norm_prices.index,
        norm_prices[col],
        linewidth=1.25,
        label=col,
        zorder=2
    )

# No title inside the figure; use LaTeX caption instead
ax.set_xlabel("Date")
ax.set_ylabel("Index level, rebased to 100")

# Report-style clean formatting
ax.grid(False)
ax.legend(ncol=3, frameon=False)

fig.tight_layout()
savefig(FIG_DIR / "fig_01_normalized_prices.pdf")

In [ ]:
# ============================================================
# Figure 2: Daily log returns
# Input: vietnam_size_indices_log_returns_common_20141121_20251231.csv
# Output: figures/fig_02_daily_returns.pdf
# ============================================================

ret_path = Path("data/processed/vietnam_size_indices_log_returns_common_20141121_20251231.csv")
returns_df = pd.read_csv(ret_path)
returns_df = standardize_date_index(returns_df)

ret_cols = pick_index_columns(returns_df)
returns_plot = returns_df[ret_cols].copy()
returns_plot.columns = INDEX_ORDER

fig, axes = plt.subplots(4, 1, figsize=(10, 7.2), sharex=True)

for ax, col in zip(axes, INDEX_ORDER):
    ax.plot(
        returns_plot.index,
        returns_plot[col],
        linewidth=0.65,
        zorder=2
    )

    ax.axhline(
        0,
        linewidth=0.8,
        color="black",
        alpha=0.75,
        zorder=1
    )

    # Keep individual index names as subplot titles
    ax.set_title(col, loc="center", fontsize=10, fontweight="normal")

    ax.set_ylabel("Return (%)")

    # Remove grid for report-style figure
    ax.grid(False)

axes[-1].set_xlabel("Date")

# No common/suptitle; use LaTeX caption instead
fig.tight_layout()

savefig(FIG_DIR / "fig_02_daily_returns.pdf")